# Generate reports with Gemini

This notebook was used to create reports using the Gemini API. It processes a dataset of prompts, generates responses in batches, and saves the results to a text file. The responses can be parsed with the `parse_gemini_responses.ipynb` notebook. You need your own API key to run this.

In [ ]:
from google import genai
import pandas as pd
import time
import traceback

train_dataset = pd.read_csv("train_dataset.csv")

In [ ]:
batch_size = 10
start_idx, end_idx = 0, len(train_dataset)

# Add your own API key here
client = genai.Client(api_key="XXX")

with open("results.txt", "a") as f:
    pass

i = start_idx
while i < end_idx:
    try:
        batched_prompts = []
        for j in range(batch_size):
            if i + j >= end_idx:
                break
            row = train_dataset.loc[i + j]
            prompt = row['prompt']
            
            # Don't include the report
            prompt = prompt.split("### Poročilo")[0]
            batched_prompts.append(prompt)

        # Generate content for the batch of prompts
        prompt = \
        """
        Generiraj 10 poročil o prometu na osnovi spodnjih podatkov. Vhodni podatki za vsako poročilo so ločeni z '### Vhodni podatki'. Odgovor naj vsebuje samo poročila. Odgovarjaj v povedih. Poročila loči z '### Poročilo'.
        Drži se hierarhije dogodkov (od najpomembnejših do najmanj pomembnih): 
        - Voznik vozi v napačno smer  
        - Zaprta avtocesta 
        - Nesreča z zastojem na avtocesti 
        - Zastoji zaradi del na avtocesti (ob krajših zastojih se pogosto dogajajo naleti) 
        - Zaradi nesreče zaprta glavna ali regionalna cesta 
        - Nesreče na avtocestah in drugih cestah 
        - Pokvarjena vozila, ko je zaprt vsaj en prometni pas 
        - Žival, ki je zašla na vozišče 
        - Predmet/razsut tovor na avtocesti 
        - Dela na avtocesti, kjer je večja nevarnost naleta (zaprt prometni pas, pred predori, v predorih, …) 
        - Zastoj pred Karavankami in mejnimi prehodi 
        Pomembno je sporočiti, če voznik ne vozi več v napačno smer ali če je konec zastojev zaradi katere koli prometne nesreče.
        """
        prompt += "\n\n".join(batched_prompts)
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt
        )
        
        print("Processed index " + str(i))
        i+= batch_size
        
        with open("results.txt", "a") as f:
            f.write(f"Index {i}:\n{response.text}\nEnd of index {i}\n")
        
        # Sleep to avoid hitting API limits
        time.sleep(3)
            
    except Exception:
        print(traceback.format_exc())
